<a href="https://colab.research.google.com/github/Aivon99/BigDataAndTextMiningProject/blob/main/src/eval/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Downloading Libraries and Imports

In [1]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~orch*
!pip install -q --upgrade torch torchvision torchaudio
!pip install -q --upgrade transformers>=4.45.0 accelerate torchao pillow scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub

# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
from functools import partial
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)


W0902 17:37:51.537000 13425 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0902 17:37:51.597000 13425 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Setting up environment

In [2]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}")
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

# 4. Import custom project modules cleanly
from data.generation import (
    build_sample,
    generate_dataset,
)

from data.utilities import (
    load_lichess_csv,
    upload_dataset_to_hub,
    authenticate_hf,
)

from eval.utilities import (
   calculate_fen_exact_match,
   calculate_levenshtein_metrics,
   calculate_square_by_square_accuracy,
   evaluate_chessboard_model_task_1,
   preprocess_function,
   get_patch_reordering_indices,
   reorder_chessboard_image,
   finetune_and_push_chessboard_model,
)

print("All custom modules and eval utilities imported successfully!")

Repository directory already exists at: /content/BigDataAndTextMiningProject
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: Tesla T4
All custom modules and eval utilities imported successfully!


Dowloading dataset for task 1 from HuggingFace repo

In [3]:
print("Verifying Authentication to Hugging Face...")
notebook_login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task1 = load_dataset(dataset_name)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

Verifying Authentication to Hugging Face...


Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 4
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 4
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': '2GDeK', 'task': 'task1', 'fen': 'r1b2rk1/ppRq3p/3p2p1/3PPp2/8/3B1NP1/2Q2K1P/8 b - - 1 30', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of all pieces on th

## Vanilla Model

Loading model

In [4]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model_vanilla = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model Qwen/Qwen3.5-0.8B...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model and Processor loaded correctly!


Test with baseline on a single sample

In [5]:
# Grab the first test sample
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

# Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

Sample ID: sample_000000
FEN: 4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - - 5 28
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to extract the exact board state from the provided chessboard image.
Input:
- Board Image: The visual representation of the chessboard.
Output Format:
Return only the valid FEN string representing the position of all pieces on the board.

Real FEN (Ground Truth): 4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - - 5 28

Generating zero-shot prediction...
Predicted FEN (Zero-Shot): a1b2c3d4e5f6g7h8
8h7g7e8
7d6c5b4a3
4e3d2c1b0
1f2g1h0
0d0e0f0g0h0
0a0b0c0d0e0f0g0h0
0a0b0c0d0e0f0g0h0


Testing vanilla model on the whole dataset using the function *evaluate_chessboard_model_task_1*

In [6]:
# Call the evaluation function for the vanilla model
vanilla_results_df, vanilla_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Vanilla Qwen2.5-VL (Zero-Shot)"
)

# Initialize the global comparison DataFrame with the vanilla results
all_models_results = vanilla_summary_df

print("\nExample of results obtained form the evaluation:")
display(vanilla_results_df.head())

print("\nComparative Summary DataFrame (all_models_results):")
display(all_models_results)

Evaluating Vanilla Qwen2.5-VL (Zero-Shot): 100%|██████████| 4/4 [01:06<00:00, 16.56s/it]


Evaluation completed for Vanilla Qwen2.5-VL (Zero-Shot)! Results saved to task1_vanilla_qwen2.5-vl_(zero-shot)_results.csv.

Example of results obtained form the evaluation:


,sample_id,ground_truth,predicted,fen_exact_match,levenshtein_distance,character_error_rate,square_by_square_accuracy
0,sample_000000,4Q3/p5pp/2p1p1k1/2Pp4/3n2B1/8/P2q2rP/5R1K b - ...,a1b2c3d4e5f6g7h8\n8h7g7e8\n7d6c5b4a3\n4e3d2c1b...,0.0,92,1.769231,0.0
1,sample_000001,3q1rk1/1p1bbppp/p3p3/1n1pP3/3N1P2/3QB2P/PPB3P1...,a1b2b3b4b5b6b7b8c8d8d9d9e9e1e1f1f2f3f4f5f6f7g7...,0.0,58,0.920635,0.0
2,sample_000002,8/2Kbk3/1B1p4/2pPp3/2B1Pp2/pP6/Pr3R2/8 w - - 1 51,a2 b2 c2 d2 e2 f2 g2 h2\na3 b3 c3 d3 e3 f3 g3 ...,0.0,133,2.714286,0.0
3,sample_000003,6k1/4bppp/2b1p3/1pNpP3/3P4/P3B3/r4PPP/2R3K1 w ...,a1b2b3b4b5b6b7b8c8d8e8f8g8h8,0.0,48,0.888889,0.0



Comparative Summary DataFrame (all_models_results):


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,1.57326,0.0,82.75


## Vanilla + LoRA

Preprocessing dataset and finetuning

In [7]:
# 1. Preprocessing dataset
print("Applying preprocessing to datasets...")
tokenized_train = dataset_task1["train"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task1["train"].column_names,
)
tokenized_val = dataset_task1["validation"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task1["validation"].column_names,
)

# 2. Configure PEFT and LoRA parameters
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

# Apply LoRA to model vanilla and saving the new one into lora_model
lora_model = get_peft_model(model_vanilla, peft_config)
lora_model.print_trainable_parameters()

# 3. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen_task1_lora_output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=2,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    remove_unused_columns=False,
    report_to="none",
)

# 4. Initialize the Trainer using 'lora_model'
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

# 5. Start Fine-Tuning
print("Starting LoRA Supervised Fine-Tuning...")
trainer.train()

Applying preprocessing to datasets...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435
Starting LoRA Supervised Fine-Tuning...


Epoch,Training Loss,Validation Loss
1,No log,15.430305
2,No log,14.758148


TrainOutput(global_step=8, training_loss=15.681829452514648, metrics={'train_runtime': 102.2583, 'train_samples_per_second': 0.626, 'train_steps_per_second': 0.078, 'total_flos': 118618822017024.0, 'train_loss': 15.681829452514648, 'epoch': 2.0})

Saving model on hugging face

In [8]:
# 6. Save weights to Hugging Face folder
hf_org_prefix = "bdatm-project"
repo_id_standard = f"{hf_org_prefix}/qwen-task1-standard-lora"

print(f"Pushing standard LoRA model and processor to Hugging Face Hub: {repo_id_standard}...")

trainer.model.push_to_hub(
    repo_id_standard,
    commit_message="Training complete for standard LoRA baseline (raster-scan)"
)
processor.push_to_hub(
    repo_id_standard
)

print("Fine-tuning completed and weights successfully uploaded to Hugging Face Hub!")

Pushing standard LoRA model and processor to Hugging Face Hub: bdatm-project/qwen-task1-standard-lora...


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 18.0kB / 25.6MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpx3xmjyu2/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Fine-tuning completed and weights successfully uploaded to Hugging Face Hub!


Loading model and evaluation

In [9]:
# 7. Loading Model from hugging face and evaluate model using predefined functions
print("\nLoading standard LoRA model from Hugging Face for evaluation...")

# Load fresh base model instance
base_eval_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Load the LoRA weights directly from the Hub repository
standard_lora_eval_model = PeftModel.from_pretrained(base_eval_model, repo_id_standard)

print("Evaluating the Hugging Face LoRA model on the test set...")
lora_results_df, lora_summary_df = evaluate_chessboard_model_task_1(
    model=standard_lora_eval_model,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Qwen + LoRA Fine-Tuning (from HF)"
)

# Append the new metrics to the global comparison DataFrame
all_models_results = pd.concat([all_models_results, lora_summary_df], ignore_index=True)

print("\nUpdated Comparative Summary Table (all_models_results):")
display(all_models_results)


Loading standard LoRA model from Hugging Face for evaluation...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Evaluating the Hugging Face LoRA model on the test set...


Evaluating Qwen + LoRA Fine-Tuning (from HF): 100%|██████████| 4/4 [00:50<00:00, 12.60s/it]


Evaluation completed for Qwen + LoRA Fine-Tuning (from HF)! Results saved to task1_qwen_+_lora_fine-tuning_(from_hf)_results.csv.

Updated Comparative Summary Table (all_models_results):


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,1.573260,0.0,82.75
1,Qwen + LoRA Fine-Tuning (from HF),0.0,1.949023,0.0,108.75


## Reordering patches - Advanced models

Checking function *get_patch_reordering_indices()*

In [10]:
for strat in ["raster", "zigzag", "spiral", "file_wise"]:
    order_map = get_patch_reordering_indices(strategy=strat)
    print(f"Strategy '{strat}' first 10 patch indices: {order_map[:10]}")

Strategy 'raster' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Strategy 'zigzag' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 15, 14]
Strategy 'spiral' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 15, 23]
Strategy 'file_wise' first 10 patch indices: [0, 8, 16, 24, 32, 40, 48, 56, 1, 9]


Let's evalaute the model we created before using three different patch ordering: zigzag, spiral and file-wise. We'll use the function ***reorder_chessboard_image*** defined in *src/eval/utilities.py*

In [11]:
# ==========================================
# Training-Free Benchmark Loop for REOrder
# ==========================================

# Define the strategies you want to benchmark (as outlined in the project specs)
strategies_to_test = ["zigzag", "spiral", "file_wise"]

# Choose the model to test (we use the fine-tuned LoRA model)
model_to_evaluate = lora_model  # You can switch to 'model' if you want to test the vanilla version

for strat in strategies_to_test:
    print(f"\nEvaluating strategy (Training-Free): {strat.upper()}...")

    # 1. Apply the reordering function to the images in the test set
    reordered_test_split = dataset_task1["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 2. Run the evaluation function on the reordered test split
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_1(
        model=model_to_evaluate,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - TF)"
    )

    # 3. Append the results to your global comparison table
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table (Including Training-Free Strategies) ---")
display(all_models_results)


Evaluating strategy (Training-Free): ZIGZAG...


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Evaluating Qwen + LoRA (Zigzag - TF): 100%|██████████| 4/4 [01:06<00:00, 16.59s/it]


Evaluation completed for Qwen + LoRA (Zigzag - TF)! Results saved to task1_qwen_+_lora_(zigzag_-_tf)_results.csv.

Evaluating strategy (Training-Free): SPIRAL...


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

...


Evaluation completed for Qwen + LoRA (Spiral - TF)! Results saved to task1_qwen_+_lora_(spiral_-_tf)_results.csv.

Evaluating strategy (Training-Free): FILE_WISE...


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Evaluating Qwen + LoRA (File_wise - TF): 100%|██████████| 4/4 [01:07<00:00, 16.79s/it]


Evaluation completed for Qwen + LoRA (File_wise - TF)! Results saved to task1_qwen_+_lora_(file_wise_-_tf)_results.csv.

--- Final Comparative Summary Table (Including Training-Free Strategies) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,1.573260,0.0,82.75
1,Qwen + LoRA Fine-Tuning (from HF),0.0,1.949023,0.0,108.75
2,Qwen + LoRA (Zigzag - TF),0.0,2.399369,0.0,129.75
3,Qwen + LoRA (Spiral - TF),0.0,2.236471,0.0,120.75
4,Qwen + LoRA (File_wise - TF),0.0,2.063972,0.0,111.50


As highlighted in the summary table, applying unconventional patch reordering strategies (such as Zigzag, Spiral, or File-wise) in a "Training-Free" (TF) manner leads to a performance drop, resulting in an increased Character Error Rate (CER) and Levenshtein distance compared to the standard raster-scan baseline.

To truly reap the benefits of the REOrder methodology, we must proceed with Supervised Fine-Tuning (SFT) directly on the pre-reordered dataset. This will allow the model to adapt its weights and attention layers to the new spatial serialization strategy.

Finetuning the new models on the dataset using function **finetune_and_push_chessboard_model()** defined in *src/eval/utilities.py*. This function directly upload the models on hugging face

In [12]:
strategies_to_train = ["zigzag", "spiral", "file_wise"]

# Dictionary to store the trained models in memory
trained_reordered_models = {}

for strat in strategies_to_train:
    trained_reordered_models[strat] = finetune_and_push_chessboard_model(
        strategy_name=strat,
        dataset=dataset_task1,
        processor=processor,
        peft_config=peft_config,
        task="task1",
    )

print("\nAll reordered models have been successfully trained and pushed to Hugging Face!")


Starting pipeline for TASK: TASK1 | STRATEGY: ZIGZAG
Applying zigzag reordering to datasets for task1...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Preprocessing datasets...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpyrt8xfg8/tokenizer.json:  80%|#######9  | 16.0MB / 20.0MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Finished! Successfully uploaded to Hub: bdatm-project/qwen-task1-file_wise-lora

All reordered models have been successfully trained and pushed to Hugging Face!


Evaluating models using function **evaluate_chessboard_model_task_1()** defined in *src/eval/utilities.py*.

Models are downloaded from the hugging face repo.

In [13]:
strategies_to_evaluate = ["zigzag", "spiral", "file_wise"]
hf_org_prefix = "bdatm-project"  # Assicurati che corrisponda al prefisso usato per il push

for strat in strategies_to_evaluate:
    print(f"\nLoading and evaluating SFT model for strategy: {strat.upper()} from Hugging Face...")

    # 1. Load fresh base model instance
    base_eval_model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3.5-0.8B",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # 2. Load the specific LoRA weights from Hugging Face Hub
    repo_id_source = f"{hf_org_prefix}/qwen-task1-{strat}-lora"
    model_to_eval = PeftModel.from_pretrained(base_eval_model, repo_id_source)

    # 3. Apply the specific patch reordering to the test split images
    reordered_test_split = dataset_task1["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 4. Run the evaluation utility function
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_1(
        model=model_to_eval,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - SFT)"
    )

    # 5. Append results to the global comparison DataFrame
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table (Loaded from Hub & Evaluated) ---")
display(all_models_results)


Loading and evaluating SFT model for strategy: ZIGZAG from Hugging Face...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Evaluating Qwen + LoRA (File_wise - SFT): 100%|██████████| 4/4 [01:06<00:00, 16.64s/it]


Evaluation completed for Qwen + LoRA (File_wise - SFT)! Results saved to task1_qwen_+_lora_(file_wise_-_sft)_results.csv.

--- Final Comparative Summary Table (Loaded from Hub & Evaluated) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,1.573260,0.0,82.75
1,Qwen + LoRA Fine-Tuning (from HF),0.0,1.949023,0.0,108.75
2,Qwen + LoRA (Zigzag - TF),0.0,2.399369,0.0,129.75
3,Qwen + LoRA (Spiral - TF),0.0,2.236471,0.0,120.75
4,Qwen + LoRA (File_wise - TF),0.0,2.063972,0.0,111.50
5,Qwen + LoRA (Zigzag - SFT),0.0,2.181777,0.0,118.00
6,Qwen + LoRA (Spiral - SFT),0.0,2.236471,0.0,120.75
7,Qwen + LoRA (File_wise - SFT),0.0,2.298099,0.0,126.25


Displaying oredered results

In [14]:
sorted_results_df = all_models_results.sort_values(by="character_error_rate", ascending=True).reset_index(drop=True)

print("\n--- Comparative Summary Table (Ordered from Best to Worst by CER) ---")
display(sorted_results_df)


--- Comparative Summary Table (Ordered from Best to Worst by CER) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,1.573260,0.0,82.75
1,Qwen + LoRA Fine-Tuning (from HF),0.0,1.949023,0.0,108.75
2,Qwen + LoRA (File_wise - TF),0.0,2.063972,0.0,111.50
3,Qwen + LoRA (Zigzag - SFT),0.0,2.181777,0.0,118.00
4,Qwen + LoRA (Spiral - SFT),0.0,2.236471,0.0,120.75
5,Qwen + LoRA (Spiral - TF),0.0,2.236471,0.0,120.75
6,Qwen + LoRA (File_wise - SFT),0.0,2.298099,0.0,126.25
7,Qwen + LoRA (Zigzag - TF),0.0,2.399369,0.0,129.75
